# Week 5 Lab: Bytown NetOps API Toolkit
**NET2008 DevOps · Algonquin College**

This lab is graded (1% of final grade). It is the hands-on companion to the Week 5 lecture, and it covers the same ground in more detail and with more examples: **data formats (JSON, XML, YAML)** and **calling REST APIs with `requests`**.

**How this notebook is organized.** Each part follows the same rhythm:

1. a short **"Worked example"** (code that already works: run it, read the output, change a value and run it again);
2. a **"Warm-up"** with its expected output, so you can check your understanding before the graded task (not graded);
3. a **graded task** (`Q1` to `Q13`) with a `# your code here` section, followed by a **self-check** cell. The self-check tells you whether your answer has the right shape and type. It does **not** replace reading your own output: your instructor checks the actual values against your personal `seed`.

Work top to bottom and run each cell as you finish it. Later tasks reuse helpers and variables defined earlier.

| # | Task | Points | Focus |
|---|------|-------:|-------|
| 1 | Interface table | 8 | JSON parsing and navigation |
| 2 | Snapshot file | 7 | JSON files, `default=`, types that do not round-trip |
| 3 | Defensive parsing | 6 | `JSONDecodeError`, returning errors instead of crashing |
| 4 | Firewall rules | 7 | XML parsing, attributes, text-to-number conversion |
| 5 | Scan report | 7 | Building and writing XML, escaping |
| 6 | The YAML type trap | 7 | YAML scalars, quoting, `safe_load` |
| 7 | Format conversion | 6 | YAML, JSON, XML conversion and anchors |
| 8 | Query the inventory | 7 | `GET`, `params=`, status codes |
| 9 | Device lifecycle | 9 | `POST`, `PATCH`, `PUT`, `DELETE` |
| 10 | Errors and authentication | 7 | 4xx codes, tokens, timeouts, exceptions |
| 11 | Retry and rate limits | 9 | Backoff, `Retry-After`, transient vs permanent errors |
| 12 | Pagination | 8 | Following `next` links, generators |
| 13 | Audit capstone | 12 | A client class, all three formats |

**Suggested pacing (3 hours):** Part 1 JSON 30 min · Part 2 XML 25 min · Part 3 YAML 25 min · Part 4 requests basics 35 min · Part 5 robustness 30 min · Capstone 25 min · submit 10 min.

**Before you start:** have the Week 5 lecture notebook open. Section numbers below (for example "lecture 15.7") point back to it.

## 0. Personalize and set up

**Personalization.** Set `your_name` and `seed` in the cell below. `seed` = the sum of the **last four digits of your student ID** (for example ID `040112348` gives `2+3+4+8 = 17`). Several tasks use values derived from `seed`, so your numeric answers must match your own ID.

In [ ]:
your_name = ""   # e.g. "Amina Khoury"
seed = 0         # sum of the last four digits of YOUR student ID
assert your_name.strip() and 0 < seed <= 36, "Fill in your_name and a valid seed first."
print(f"{your_name} · seed={seed}")

**Setup.** The next cells install anything missing, write the practice API (`mockapi.py`) to disk, and start it inside this notebook. It is the same service as in the lecture (see lecture section 19.7 for the list of endpoints), so every example works offline and every student sees the same responses. You do not need to read or edit the server code.

In [ ]:
import importlib, subprocess, sys

for module, pip_name in [("requests", "requests"), ("yaml", "pyyaml")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

import json, os, time
import xml.etree.ElementTree as ET
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import requests, yaml
print("requests", requests.__version__, "| PyYAML", yaml.__version__, "| Python", sys.version.split()[0])

In [ ]:
%%writefile mockapi.py
"""Mock REST API for the Week 5 lecture and lab (NET2008 DevOps).

A tiny device-inventory service built on the standard library only. It runs
inside the notebook in a background thread, so every example works offline and
every student sees exactly the same responses.

Endpoints (all under http://127.0.0.1:<port>):

    GET    /api/devices                list, paginated (?page=1&per_page=5, ?status=up)
    GET    /api/devices/<id>           one device (404 if unknown)
    POST   /api/devices                create (201; 400 on bad JSON; 422 on missing fields)
    PUT    /api/devices/<id>           replace (200)
    PATCH  /api/devices/<id>           partial update (200)
    DELETE /api/devices/<id>           delete (204)
    GET    /api/devices.xml            same inventory as XML
    GET    /api/devices.yaml           same inventory as YAML
    GET    /api/secure/ping            needs  Authorization: Bearer netops-token
    GET    /api/flaky                  503 twice, then 200 (resets via /api/reset)
    GET    /api/slow                   answers after 2 seconds
    GET    /api/limited                429 + Retry-After on the 3rd call in a row
    GET    /api/echo                   returns the query string, headers it received
    POST   /api/reset                  restore the seed data and all counters
"""
import copy
import json
import threading
import time
import xml.etree.ElementTree as ET
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse, parse_qs

TOKEN = "netops-token"

SEED_DEVICES = [
    {"id": 1, "hostname": "OTT-CORE-01", "ip": "10.10.1.1", "type": "router", "ports": 8, "status": "up", "tags": ["core", "ospf"]},
    {"id": 2, "hostname": "OTT-CORE-02", "ip": "10.10.1.2", "type": "router", "ports": 8, "status": "up", "tags": ["core", "bgp"]},
    {"id": 3, "hostname": "OTT-SW-01", "ip": "10.10.2.1", "type": "switch", "ports": 48, "status": "up", "tags": ["access"]},
    {"id": 4, "hostname": "OTT-SW-02", "ip": "10.10.2.2", "type": "switch", "ports": 48, "status": "down", "tags": ["access"]},
    {"id": 5, "hostname": "OTT-SW-03", "ip": "10.10.2.3", "type": "switch", "ports": 24, "status": "up", "tags": ["access", "lab"]},
    {"id": 6, "hostname": "TOR-FW-01", "ip": "10.20.1.1", "type": "firewall", "ports": 12, "status": "up", "tags": ["edge"]},
    {"id": 7, "hostname": "TOR-FW-02", "ip": "10.20.1.2", "type": "firewall", "ports": 12, "status": "down", "tags": ["edge"]},
    {"id": 8, "hostname": "TOR-SW-01", "ip": "10.20.2.1", "type": "switch", "ports": 48, "status": "up", "tags": ["access"]},
    {"id": 9, "hostname": "MTL-RTR-01", "ip": "10.30.1.1", "type": "router", "ports": 4, "status": "up", "tags": ["branch"]},
    {"id": 10, "hostname": "MTL-SW-01", "ip": "10.30.2.1", "type": "switch", "ports": 24, "status": "up", "tags": ["branch"]},
    {"id": 11, "hostname": "MTL-AP-01", "ip": "10.30.3.1", "type": "access-point", "ports": 2, "status": "down", "tags": ["branch", "wifi"]},

]

_lock = threading.Lock()
_state = {}


def _reset():
    _state["devices"] = copy.deepcopy(SEED_DEVICES)
    _state["next_id"] = max(d["id"] for d in SEED_DEVICES) + 1
    _state["flaky_calls"] = 0
    _state["limited_calls"] = 0


_reset()


def devices_to_xml(devices):
    root = ET.Element("inventory", site="bytown")
    for d in devices:
        el = ET.SubElement(root, "device", id=str(d["id"]), type=d["type"])
        ET.SubElement(el, "hostname").text = d["hostname"]
        ET.SubElement(el, "ip").text = d["ip"]
        ET.SubElement(el, "ports").text = str(d["ports"])
        ET.SubElement(el, "status").text = d["status"]
        tags = ET.SubElement(el, "tags")
        for t in d["tags"]:
            ET.SubElement(tags, "tag").text = t
    return ET.tostring(root, encoding="unicode")


def devices_to_yaml(devices):
    lines = ["site: bytown", "devices:"]
    for d in devices:
        lines.append(f"  - id: {d['id']}")
        lines.append(f"    hostname: {d['hostname']}")
        lines.append(f"    ip: {d['ip']}")
        lines.append(f"    type: {d['type']}")
        lines.append(f"    ports: {d['ports']}")
        lines.append(f"    status: {d['status']}")
        lines.append("    tags: [" + ", ".join(d["tags"]) + "]")
    return "\n".join(lines) + "\n"


class Handler(BaseHTTPRequestHandler):
    server_version = "BytownMock/1.0"
    protocol_version = "HTTP/1.1"

    def log_message(self, fmt, *args):  # keep the notebook output clean
        pass

    # ---- helpers -------------------------------------------------------
    def _send(self, code, body=None, ctype="application/json", headers=None):
        if body is None:
            payload = b""
        elif isinstance(body, (dict, list)):
            payload = json.dumps(body).encode("utf-8")
        elif isinstance(body, str):
            payload = body.encode("utf-8")
        else:
            payload = body
        self.send_response(code)
        if payload:
            self.send_header("Content-Type", ctype)
        self.send_header("Content-Length", str(len(payload)))
        for k, v in (headers or {}).items():
            self.send_header(k, v)
        self.end_headers()
        if payload:
            try:
                self.wfile.write(payload)
            except (BrokenPipeError, ConnectionResetError):
                pass                     # the client gave up (for example after a timeout)

    def _error(self, code, message, headers=None):
        self._send(code, {"error": message, "status": code}, headers=headers)

    def _read_json(self):
        length = int(self.headers.get("Content-Length") or 0)
        raw = self.rfile.read(length) if length else b""
        try:
            return json.loads(raw.decode("utf-8")), None
        except (ValueError, UnicodeDecodeError):
            return None, "request body is not valid JSON"

    def _route(self):
        parsed = urlparse(self.path)
        return parsed.path.rstrip("/") or "/", parse_qs(parsed.query)

    def _device_id(self, path):
        parts = path.split("/")
        if len(parts) == 4 and parts[1] == "api" and parts[2] == "devices" and parts[3].isdigit():
            return int(parts[3])
        return None

    def _find(self, dev_id):
        for d in _state["devices"]:
            if d["id"] == dev_id:
                return d
        return None

    # ---- verbs ---------------------------------------------------------
    def do_GET(self):
        path, q = self._route()
        with _lock:
            if path == "/api/devices":
                items = list(_state["devices"])
                if "status" in q:
                    items = [d for d in items if d["status"] == q["status"][0]]
                if "type" in q:
                    items = [d for d in items if d["type"] == q["type"][0]]
                try:
                    page = int(q.get("page", ["1"])[0])
                    per_page = int(q.get("per_page", ["5"])[0])
                except ValueError:
                    return self._error(400, "page and per_page must be integers")
                if page < 1 or per_page < 1:
                    return self._error(400, "page and per_page must be >= 1")
                total = len(items)
                start = (page - 1) * per_page
                chunk = items[start:start + per_page]
                host = self.headers.get("Host", "127.0.0.1")
                nxt = None
                if start + per_page < total:
                    nxt = f"http://{host}/api/devices?page={page + 1}&per_page={per_page}"
                    if "status" in q:
                        nxt += f"&status={q['status'][0]}"
                return self._send(200, {"page": page, "per_page": per_page, "total": total, "next": nxt, "results": chunk})

            dev_id = self._device_id(path)
            if dev_id is not None:
                d = self._find(dev_id)
                return self._send(200, d) if d else self._error(404, f"device {dev_id} not found")

            if path == "/api/devices.xml":
                return self._send(200, devices_to_xml(_state["devices"]), "application/xml")
            if path == "/api/devices.yaml":
                return self._send(200, devices_to_yaml(_state["devices"]), "application/yaml")

            if path == "/api/secure/ping":
                auth = self.headers.get("Authorization", "")
                if not auth:
                    return self._error(401, "missing Authorization header", {"WWW-Authenticate": "Bearer"})
                if auth != f"Bearer {TOKEN}":
                    return self._error(403, "token not accepted")
                return self._send(200, {"pong": True, "user": "netops"})

            if path == "/api/flaky":
                _state["flaky_calls"] += 1
                if _state["flaky_calls"] <= 2:
                    return self._error(503, f"temporarily unavailable (call {_state['flaky_calls']})")
                return self._send(200, {"ok": True, "calls_needed": _state["flaky_calls"]})

            if path == "/api/limited":
                _state["limited_calls"] += 1
                if _state["limited_calls"] % 3 == 0:
                    return self._error(429, "slow down", {"Retry-After": "1"})
                return self._send(200, {"ok": True, "call": _state["limited_calls"]})

            if path == "/api/echo":
                return self._send(200, {"query": {k: v if len(v) > 1 else v[0] for k, v in q.items()},
                                        "user_agent": self.headers.get("User-Agent"),
                                        "accept": self.headers.get("Accept")})

        if path == "/api/slow":
            time.sleep(2)
            return self._send(200, {"ok": True, "waited": 2})
        return self._error(404, f"no route for GET {path}")

    def do_POST(self):
        path, _ = self._route()
        if path == "/api/reset":
            with _lock:
                _reset()
            return self._send(200, {"reset": True})
        if path == "/api/devices":
            body, err = self._read_json()
            if err:
                return self._error(400, err)
            if not isinstance(body, dict):
                return self._error(400, "body must be a JSON object")
            missing = [k for k in ("hostname", "ip", "type") if not body.get(k)]
            if missing:
                return self._error(422, "missing fields: " + ", ".join(missing))
            with _lock:
                dev = {"id": _state["next_id"], "hostname": body["hostname"], "ip": body["ip"], "type": body["type"],
                       "ports": body.get("ports", 0), "status": body.get("status", "up"), "tags": body.get("tags", [])}
                _state["next_id"] += 1
                _state["devices"].append(dev)
            return self._send(201, dev, headers={"Location": f"/api/devices/{dev['id']}"})
        return self._error(404, f"no route for POST {path}")

    def do_PUT(self):
        path, _ = self._route()
        dev_id = self._device_id(path)
        if dev_id is None:
            return self._error(404, f"no route for PUT {path}")
        body, err = self._read_json()
        if err:
            return self._error(400, err)
        with _lock:
            d = self._find(dev_id)
            if not d:
                return self._error(404, f"device {dev_id} not found")
            missing = [k for k in ("hostname", "ip", "type") if not isinstance(body, dict) or not body.get(k)]
            if missing:
                return self._error(422, "PUT replaces the whole record; missing: " + ", ".join(missing))
            d.clear()
            d.update({"id": dev_id, "hostname": body["hostname"], "ip": body["ip"], "type": body["type"],
                      "ports": body.get("ports", 0), "status": body.get("status", "up"), "tags": body.get("tags", [])})
            return self._send(200, d)

    def do_PATCH(self):
        path, _ = self._route()
        dev_id = self._device_id(path)
        if dev_id is None:
            return self._error(404, f"no route for PATCH {path}")
        body, err = self._read_json()
        if err:
            return self._error(400, err)
        with _lock:
            d = self._find(dev_id)
            if not d:
                return self._error(404, f"device {dev_id} not found")
            if not isinstance(body, dict):
                return self._error(400, "body must be a JSON object")
            for k, v in body.items():
                if k != "id":
                    d[k] = v
            return self._send(200, d)

    def do_DELETE(self):
        path, _ = self._route()
        dev_id = self._device_id(path)
        if dev_id is None:
            return self._error(404, f"no route for DELETE {path}")
        with _lock:
            d = self._find(dev_id)
            if not d:
                return self._error(404, f"device {dev_id} not found")
            _state["devices"].remove(d)
            return self._send(204)


class QuietServer(ThreadingHTTPServer):
    daemon_threads = True

    def handle_error(self, request, client_address):   # no tracebacks in the notebook output
        pass


_server = None


def start(port=0):
    """Start the server in a daemon thread and return its base URL."""
    global _server
    if _server is None:
        _server = QuietServer(("127.0.0.1", port), Handler)
        threading.Thread(target=_server.serve_forever, daemon=True).start()
    return f"http://127.0.0.1:{_server.server_address[1]}"


def stop():
    global _server
    if _server is not None:
        _server.shutdown()
        _server.server_close()
        _server = None


def reset():
    with _lock:
        _reset()

In [ ]:
import mockapi

BASE = mockapi.start()
TOKEN = "netops-token"
os.environ.setdefault("NETOPS_TOKEN", TOKEN)   # stands in for a token you would normally export in your shell
work = Path("w5lab")
work.mkdir(exist_ok=True)               # all files this lab creates go in this folder

def reset_api():
    """Restore the practice API to its starting state (11 devices, counters at zero)."""
    requests.post(f"{BASE}/api/reset", timeout=5).raise_for_status()

def selfcheck(label, *tests):
    """Run (description, function) pairs. Prints which checks failed; never raises."""
    failed = []
    for description, fn in tests:
        try:
            ok = bool(fn())
        except Exception as e:                     # an unfinished task should report, not crash
            ok, description = False, f"{description} (raised {type(e).__name__})"
        if not ok:
            failed.append(description)
    if failed:
        print(f"{label} self-check: {len(failed)} of {len(tests)} checks FAILED")
        for d in failed:
            print("   -", d)
    else:
        print(f"{label} self-check: all {len(tests)} checks passed")

reset_api()
print("practice API running at", BASE, "| devices:", requests.get(f"{BASE}/api/devices", timeout=5).json()["total"])

---
# Part 1. JSON

Quick reminders from lecture sections 15.1 to 15.8:

- `json.loads(text)` / `json.dumps(obj)` work on **strings**; `json.load(f)` / `json.dump(obj, f)` work on **files**.
- Parsed JSON is plain Python: objects become `dict`, arrays become `list`, `true`/`false`/`null` become `True`/`False`/`None`.
- Tuples come back as lists, integer keys come back as strings, and `datetime`/`set` need help.
- Invalid text raises `json.JSONDecodeError` (a `ValueError`).

### Worked example 1.1: reading a nested document
Read the output line by line against the document, so you can see how each Python expression walks down one level.

In [ ]:
router_doc = json.loads("""
{
  "hostname": "OTT-CORE-01",
  "software": {"vendor": "ExampleOS", "version": "17.3.1"},
  "interfaces": [
    {"name": "Gi0/0", "ip": "10.10.1.1", "enabled": true,  "vlans": [10, 20]},
    {"name": "Gi0/1", "ip": null,        "enabled": false, "vlans": []},
    {"name": "Gi0/2", "ip": "10.10.9.1", "enabled": true,  "vlans": [30]}
  ]
}
""")

print("1. top-level keys     :", list(router_doc))
print("2. nested object      :", router_doc["software"]["version"])
print("3. list length        :", len(router_doc["interfaces"]))
print("4. last interface     :", router_doc["interfaces"][-1]["name"])
print("5. vlans of first     :", router_doc["interfaces"][0]["vlans"])
print("6. interfaces with IPs:", [i["name"] for i in router_doc["interfaces"] if i["ip"] is not None])
print("7. all vlans, flat    :", [v for i in router_doc["interfaces"] for v in i["vlans"]])
print("8. enabled count      :", sum(1 for i in router_doc["interfaces"] if i["enabled"]))
print("9. optional key       :", router_doc.get("location", "not recorded"))

### Worked example 1.2: output options
Compare the same data in compact form, pretty form, and sorted form. Sorted pretty output is what you want when saving JSON in Git, because a reordered dictionary then does not show up as a change.

In [ ]:
tiny = {"b": [3, 2, 1], "a": {"y": None, "x": True}}
print("compact       :", json.dumps(tiny))
print("custom compact:", json.dumps(tiny, separators=(",", ":")))
print("indent=2      :")
print(json.dumps(tiny, indent=2))
print("sort_keys     :", json.dumps(tiny, sort_keys=True))

### Warm-up 1 (not graded)
Using `router_doc` above, write one expression each for:

- the name of the interface that has more than one VLAN
- the total number of VLAN assignments across all interfaces
- a dictionary `{interface name: ip}` for interfaces that **have** an IP

**Expected output**
```
Gi0/0
3
{'Gi0/0': '10.10.1.1', 'Gi0/2': '10.10.9.1'}
```

In [ ]:
# warm-up: your code here

### Q1 · Interface table *(8 pts)*
*Focus: parsing and navigating JSON.*

A device returned the interface table below. Parse it with `json.loads` into `table`, then compute, using the variable names given:

- `iface_count`: the number of interfaces
- `down_names`: a list of the names of interfaces whose `state` is `"down"`, in the order they appear
- `total_mbps`: total traffic (`in` plus `out`, over **all** interfaces), rounded to 2 decimals
- `busiest`: the **name** of the interface with the highest `in + out`
- `over_threshold`: sorted list of names whose `in + out` is **greater than** `threshold`, where `threshold = seed * 20`
- `utilization`: a dict `{name: percent}` for interfaces whose state is `"up"`, where percent is `100 * (in + out) / speed_mbps` rounded to 1 decimal

Print each variable on its own line, labelled.

In [ ]:
interfaces_json = """
{
  "device": "OTT-CORE-01",
  "polled_at": "2026-10-01T09:00:00Z",
  "interfaces": [
    {"name": "Gi0/0", "state": "up",   "speed_mbps": 1000,  "traffic": {"in": 420.5,  "out": 310.0}},
    {"name": "Gi0/1", "state": "down", "speed_mbps": 1000,  "traffic": {"in": 0.0,    "out": 0.0}},
    {"name": "Gi0/2", "state": "up",   "speed_mbps": 1000,  "traffic": {"in": 35.25,  "out": 640.0}},
    {"name": "Gi0/3", "state": "up",   "speed_mbps": 100,   "traffic": {"in": 12.0,   "out": 8.5}},
    {"name": "Gi0/4", "state": "down", "speed_mbps": 100,   "traffic": {"in": 0.0,    "out": 0.0}},
    {"name": "Te1/0", "state": "up",   "speed_mbps": 10000, "traffic": {"in": 2210.0, "out": 1875.5}}
  ]
}
"""
threshold = seed * 20

In [ ]:
# your code here
table = None
iface_count = None
down_names = None
total_mbps = None
busiest = None
over_threshold = None
utilization = None

for label, value in [("iface_count", iface_count), ("down_names", down_names), ("total_mbps", total_mbps),
                     ("busiest", busiest), ("over_threshold", over_threshold), ("utilization", utilization)]:
    print(f"{label:<15}", value)

In [ ]:
selfcheck("Q1",
    ("table is a dict with an 'interfaces' list", lambda: isinstance(table, dict) and isinstance(table["interfaces"], list)),
    ("iface_count is an int", lambda: isinstance(iface_count, int)),
    ("down_names is a list of str", lambda: isinstance(down_names, list) and all(isinstance(n, str) for n in down_names)),
    ("total_mbps is a float with at most 2 decimals", lambda: isinstance(total_mbps, float) and round(total_mbps, 2) == total_mbps),
    ("busiest is one of the interface names", lambda: busiest in {i["name"] for i in table["interfaces"]}),
    ("over_threshold is sorted", lambda: over_threshold == sorted(over_threshold)),
    ("utilization has only 'up' interfaces", lambda: set(utilization) <= {i["name"] for i in table["interfaces"] if i["state"] == "up"}),
)

### Worked example 1.3: types JSON cannot carry
Study the pattern: a function passed as `default=` is called **only** for objects the encoder does not understand, and must return something it does. Q2 uses exactly this pattern.

In [ ]:
from datetime import date

def encode_extra(obj):
    if isinstance(obj, (datetime, date)):
        return obj.isoformat()
    if isinstance(obj, set):
        return sorted(obj)
    if isinstance(obj, bytes):
        return obj.hex()
    raise TypeError(f"cannot serialize {type(obj).__name__}")

sample = {"when": datetime(2026, 10, 1, 9, 0, tzinfo=timezone.utc), "day": date(2026, 10, 1),
          "seen": {"b", "a", "c"}, "mac": bytes([0, 17, 34, 51, 68, 85])}
text = json.dumps(sample, default=encode_extra, indent=2)
print(text)

back = json.loads(text)
print()
print("types after loading:", {k: type(v).__name__ for k, v in back.items()})
print("restore datetime   :", datetime.fromisoformat(back["when"]))
print("restore bytes      :", bytes.fromhex(back["mac"]))

### Q2 · Snapshot file *(7 pts)*
*Focus: writing and reading JSON files, `default=`, types that do not round-trip.*

Build this dictionary named `snapshot` (using your own `your_name` and `seed`):

```python
{
  "checked_by": your_name,
  "checked_at": datetime(2026, 10, 1, 9, 0, tzinfo=timezone.utc),
  "devices_ok": {"OTT-SW-01", "OTT-SW-02", "OTT-SW-03"},     # a set
  "thresholds": (seed, seed * 2, seed * 3),                    # a tuple
  "notes": None,
}
```

1. Write an `encode_snapshot(obj)` function for `default=` that handles `datetime` (as ISO string) and `set` (as a **sorted** list).
2. Write `snapshot` to `w5lab/snapshot.json` with `json.dump(..., indent=2)` and `encoding="utf-8"` using `with`.
3. Read it back into `loaded`. Convert `loaded["checked_at"]` back to a real `datetime`.
4. Print `type()` of `loaded["thresholds"]` and explain in the variable `tuple_answer` (a one-sentence string) what happened to the tuple and why.

In [ ]:
snapshot = {
    "checked_by": your_name,
    "checked_at": datetime(2026, 10, 1, 9, 0, tzinfo=timezone.utc),
    "devices_ok": {"OTT-SW-01", "OTT-SW-02", "OTT-SW-03"},
    "thresholds": (seed, seed * 2, seed * 3),
    "notes": None,
}
snapshot_path = work / "snapshot.json"

def encode_snapshot(obj):
    # your code here
    pass

# your code here: write snapshot_path, then read it into `loaded` and restore the datetime

loaded = None
tuple_answer = ""     # one sentence

In [ ]:
selfcheck("Q2",
    ("file exists", lambda: snapshot_path.exists()),
    ("file is valid JSON", lambda: isinstance(json.loads(snapshot_path.read_text(encoding="utf-8")), dict)),
    ("devices_ok stored as a sorted list", lambda: json.loads(snapshot_path.read_text(encoding="utf-8"))["devices_ok"] == ["OTT-SW-01", "OTT-SW-02", "OTT-SW-03"]),
    ("loaded['checked_at'] is a datetime", lambda: isinstance(loaded["checked_at"], datetime)),
    ("loaded['thresholds'] is a list", lambda: isinstance(loaded["thresholds"], list) and loaded["thresholds"] == list(snapshot["thresholds"])),
    ("tuple_answer is a non-empty sentence", lambda: isinstance(tuple_answer, str) and len(tuple_answer.split()) >= 5),
)

### Worked example 1.4: parse safely
When data comes from outside your program, expect it to be broken sometimes. The pattern: **try the risky line, catch the specific exception, return something the caller can use.** Notice that `e.lineno`, `e.colno` and `e.msg` tell you exactly where the parser gave up.

In [ ]:
def parse_or_report(text):
    try:
        return json.loads(text), None
    except json.JSONDecodeError as e:
        return None, f"line {e.lineno}, column {e.colno}: {e.msg}"

for sample in ['{"ok": true}', '{"ok": True}', "{'ok': 1}", '[1, 2,]', '']:
    data, error = parse_or_report(sample)
    print(f"{sample!r:<18} -> data={data!r:<14} error={error}")

### Q3 · Defensive parsing *(6 pts)*
*Focus: `JSONDecodeError`, returning an error instead of crashing.*

Write `safe_parse(text)` that returns a tuple `(data, error)`:

- on success: `(parsed_value, None)`;
- on invalid JSON: `(None, message)` where `message` is a string that **starts with** `"invalid JSON at line "` followed by the line number, a comma, `" column "` and the column number (for example `"invalid JSON at line 1 column 8"`);
- if `text` is valid JSON but **not a JSON object** (for example `[1, 2]` or `42`), return `(None, "expected a JSON object")`.

Then run it over `documents` and store the results as: `valid_count` (how many parsed to a dict) and `error_messages` (a list of the error strings, in order, for the ones that failed).

In [ ]:
documents = [
    '{"hostname": "sw01", "ports": 48}',
    '{"hostname": "sw02", "ports": 48,}',
    '{"hostname": sw03}',
    '[1, 2, 3]',
    '{"hostname": "sw05", "tags": ["a", "b"]}',
    '',
    '{"hostname": "sw07", "up": true}',
    '42',
]

def safe_parse(text):
    # your code here
    pass

valid_count = None
error_messages = None
# your code here: loop over documents

print(valid_count)
for m in error_messages or []:
    print(" ", m)

In [ ]:
selfcheck("Q3",
    ("valid input returns (dict, None)", lambda: safe_parse('{"a": 1}') == ({"a": 1}, None)),
    ("broken input returns (None, str)", lambda: safe_parse('{"a": }')[0] is None and safe_parse('{"a": }')[1].startswith("invalid JSON at line 1 column")),
    ("array is rejected", lambda: safe_parse("[1, 2]") == (None, "expected a JSON object")),
    ("number is rejected", lambda: safe_parse("42") == (None, "expected a JSON object")),
    ("valid_count is an int", lambda: isinstance(valid_count, int)),
    ("error_messages has one entry per failure", lambda: isinstance(error_messages, list) and len(error_messages) + valid_count == len(documents)),
)

---
# Part 2. XML

Quick reminders from lecture sections 16.1 to 16.8:

- `ET.fromstring(text)` returns the **root** element; `ET.parse(path).getroot()` does the same for a file.
- `el.tag`, `el.attrib`, `el.text`, `el.get("attr")`, `el.find()`, `el.findall()`, `el.findtext()`, `el.iter()`.
- **Everything is text.** Convert with `int(...)` before comparing or adding.
- Build with `ET.Element` and `ET.SubElement`; set `.text` to a `str`.
- Escaping (`&amp;`, `&lt;`) is automatic in Python.

### Worked example 2.1: attributes, children and XPath-style searches
Run it and match each output line to the document. Pay special attention to lines 6 and 7, which show the difference between "find one" and "find all".

In [ ]:
library = ET.fromstring("""
<monitors>
  <monitor id="m1" kind="ping"  enabled="true">
    <target>10.10.1.1</target><interval>30</interval>
  </monitor>
  <monitor id="m2" kind="http"  enabled="false">
    <target>10.10.2.1</target><interval>60</interval><path>/health</path>
  </monitor>
  <monitor id="m3" kind="ping"  enabled="true">
    <target>10.10.3.1</target><interval>10</interval>
  </monitor>
</monitors>
""")

print("1. root                 :", library.tag, len(library), "children")
print("2. first monitor attrs  :", library[0].attrib)
print("3. kinds                :", [m.get("kind") for m in library])
print("4. ping targets         :", [m.findtext("target") for m in library.findall("monitor[@kind='ping']")])
print("5. enabled (str->bool)  :", [m.get("id") for m in library if m.get("enabled") == "true"])
print("6. find returns first   :", library.find("monitor").get("id"))
print("7. findall returns list :", [m.get("id") for m in library.findall("monitor")])
print("8. optional child       :", [m.findtext("path", default="-") for m in library])
print("9. sum of intervals     :", sum(int(m.findtext("interval")) for m in library))
print("10. any depth with iter :", [t.text for t in library.iter("target")])

### Warm-up 2 (not graded)
Using `library`, produce:

- the `id` of the monitor with the smallest `interval` (remember to convert to `int`)
- a dict `{id: interval}` with intervals as `int`
- the number of monitors whose `enabled` attribute is `"false"`

**Expected output**
```
m3
{'m1': 30, 'm2': 60, 'm3': 10}
1
```

In [ ]:
# warm-up: your code here

### Q4 · Firewall rules *(7 pts)*
*Focus: XML parsing, attributes, missing elements, text-to-number conversion.*

Parse `firewall_xml` and compute, using the variable names given. Use `findtext(..., default=...)` where an element may be missing.

- `rule_count`: the number of `<rule>` elements
- `deny_names`: list of `<name>` text for rules with `action="deny"`, in document order
- `deny_ports`: sorted list of **integer** ports of the deny rules (a rule without a `<port>` counts as port `0`)
- `ports_by_id`: dict `{rule id (as int): port (as int)}` for **all** rules (missing port is `0`)
- `late_rules`: names of rules whose `id` is **greater than or equal to** `cutoff`, where `cutoff = 10 + (seed % 8) * 10`, in document order
- `firewall_name`: the `name` attribute of the root

In [ ]:
firewall_xml = """<?xml version="1.0"?>
<firewall name="TOR-FW-01">
  <rule id="10" action="allow"><name>web-https</name><port>443</port><source>any</source></rule>
  <rule id="20" action="allow"><name>ssh-admin</name><port>22</port><source>10.10.0.0/16</source></rule>
  <rule id="30" action="deny"><name>telnet</name><port>23</port><source>any</source></rule>
  <rule id="40" action="deny"><name>R&amp;D-lab-block</name><port>8080</port><source>10.20.0.0/16</source></rule>
  <rule id="50" action="allow"><name>icmp-ping</name><source>10.10.0.0/16</source></rule>
  <rule id="60" action="deny"><name>smb</name><port>445</port><source>any</source></rule>
  <rule id="70" action="allow"><name>dns</name><port>53</port><source>any</source></rule>
  <rule id="80" action="deny"><name>icmp-flood</name><source>any</source></rule>
  <rule id="90" action="allow"><name>ntp</name><port>123</port><source>10.10.0.0/16</source></rule>
</firewall>"""
cutoff = 10 + (seed % 8) * 10

In [ ]:
fw = None                  # your code here: the root element
firewall_name = None
rule_count = None
deny_names = None
deny_ports = None
ports_by_id = None
late_rules = None

for label, value in [("firewall_name", firewall_name), ("rule_count", rule_count), ("deny_names", deny_names),
                     ("deny_ports", deny_ports), ("ports_by_id", ports_by_id), ("cutoff", cutoff), ("late_rules", late_rules)]:
    print(f"{label:<14}", value)

> **Hint.** `ET.fromstring` accepts a `str` that starts with `<?xml version="1.0"?>` (as above, with no `encoding=` in it). If you ever see `ValueError: Unicode strings with encoding declaration are not supported`, remove the declaration from the string or encode the string to bytes first.

In [ ]:
selfcheck("Q4",
    ("rule_count is 9", lambda: rule_count == 9),
    ("firewall_name is a str", lambda: isinstance(firewall_name, str) and firewall_name.startswith("TOR")),
    ("deny_names are in document order and unescaped", lambda: deny_names[:2] == ["telnet", "R&D-lab-block"]),
    ("deny_ports are ints, sorted, with 0 for missing", lambda: all(isinstance(p, int) for p in deny_ports) and deny_ports == sorted(deny_ports) and 0 in deny_ports),
    ("ports_by_id has int keys and int values", lambda: all(isinstance(k, int) and isinstance(v, int) for k, v in ports_by_id.items()) and len(ports_by_id) == 9),
    ("late_rules is a list of names, one per rule with id >= cutoff", lambda: isinstance(late_rules, list) and all(isinstance(n, str) for n in late_rules) and len(late_rules) == sum(1 for k in ports_by_id if k >= cutoff)),
)

### Worked example 2.2: build, indent, write, reload
This is the exact sequence Q5 needs. Notice three details: attribute values must be strings, numbers in `.text` must be converted with `str()`, and `ET.indent` has to be called **before** writing.

In [ ]:
ticket_data = [
    {"id": 101, "title": "Replace fan in rack 4", "priority": 2, "tags": ["hardware", "dc"]},
    {"id": 102, "title": "Rotate R&D VPN keys",   "priority": 1, "tags": []},
]

root = ET.Element("tickets", count=str(len(ticket_data)))
for t in ticket_data:
    el = ET.SubElement(root, "ticket", id=str(t["id"]), priority=str(t["priority"]))
    ET.SubElement(el, "title").text = t["title"]
    tags_el = ET.SubElement(el, "tags")
    for tag in t["tags"]:
        ET.SubElement(tags_el, "tag").text = tag

ET.indent(root)
path = work / "tickets.xml"
ET.ElementTree(root).write(path, encoding="utf-8", xml_declaration=True)
print(path.read_text(encoding="utf-8"))

again = ET.parse(path).getroot()
print("reloaded:", again.get("count"), "tickets;", again.findall("ticket")[1].findtext("title"))

### Q5 · Scan report *(7 pts)*
*Focus: building XML, escaping, writing and reloading a file.*

Build an XML document from `scan_results` and write it to `w5lab/scan.xml`:

- root element `<scan>` with attributes `by` (your name) and `count` (number of hosts, as text)
- one `<host>` per entry with attributes `ip` and `up` (`"true"` or `"false"`)
- inside each `<host>`: a `<ports>` element containing one `<port>` child for every open port (text = the number), and a `<note>` element whose text is the `note` value (even though it contains `&` and `<`)
- pretty-print with `ET.indent` and write with an XML declaration and `encoding="utf-8"`

Then **reload the file** with `ET.parse` into `scan_back` and compute `open_total`: the total number of `<port>` elements in the file, and `note_back`: the note of the second host as read back.

In [ ]:
scan_results = [
    {"ip": "10.10.2.1", "up": True,  "open_ports": [22, 443],      "note": "R&D switch"},
    {"ip": "10.10.2.2", "up": True,  "open_ports": [22, 80, 8080], "note": "ports < 1024 are privileged & watched"},
    {"ip": "10.10.2.3", "up": False, "open_ports": [],             "note": "powered off"},
]
scan_path = work / "scan.xml"

# your code here: build the tree and write scan_path

scan_back = None
open_total = None
note_back = None
# your code here: reload and compute

print(scan_path.read_text(encoding="utf-8") if scan_path.exists() else "not written yet")
print(open_total, "|", note_back)

In [ ]:
selfcheck("Q5",
    ("scan.xml exists and starts with an XML declaration", lambda: scan_path.read_text(encoding="utf-8").startswith("<?xml")),
    ("root is <scan> with by and count attributes", lambda: scan_back.tag == "scan" and scan_back.get("by") == your_name and scan_back.get("count") == "3"),
    ("three <host> elements with ip and up attributes", lambda: len(scan_back.findall("host")) == 3 and all(h.get("ip") and h.get("up") in ("true", "false") for h in scan_back.findall("host"))),
    ("open_total is 5", lambda: open_total == 5),
    ("special characters survived the round trip", lambda: note_back == "ports < 1024 are privileged & watched"),
    ("file contains the escaped form", lambda: "&amp;" in scan_path.read_text(encoding="utf-8") and "&lt;" in scan_path.read_text(encoding="utf-8")),
)

---
# Part 3. YAML

Quick reminders from lecture sections 17.1 to 17.8:

- Structure is **indentation** (spaces, never tabs). `key: value` needs the space after the colon. Lists use `- item` or `[a, b]`.
- `yaml.safe_load(text)` and `yaml.safe_dump(obj, sort_keys=False)`. Never `yaml.load`.
- **Unquoted values are guessed**: `yes`/`no`/`on`/`off` become booleans, `3.10` becomes the float `3.1`, `010` becomes the integer `8`, `2026-10-01` becomes a `date`. Quote what must stay a string.
- Anchors (`&name`), aliases (`*name`) and merge keys (`<<: *name`) remove repetition.

### Worked example 3.1: reading the types
The loop prints each value with its Python type. Read the output *before* looking at the answer notes below it.

In [ ]:
sample = yaml.safe_load("""
name: edge-gateway
port: 8443
ratio: 0.75
enabled: true
disabled: false
nothing: null
also_nothing: ~
tags: [edge, tls]
limits:
  cpu: 2
  memory: 512Mi
motd: |
  Authorized use only.
  Activity is logged.
""")
for key, value in sample.items():
    print(f"{key:<13} {type(value).__name__:<6} {value!r}")

**Notes.** `nothing` and `also_nothing` are both `None`. `memory: 512Mi` is a string because `512Mi` does not look like a number. `motd` keeps its line break because of the `|`.

### Worked example 3.2: the traps, one at a time
This is the lecture's table of surprises, shown with a **fix** for each. In every row the fix is the same idea: *tell YAML it is a string by quoting it*.

In [ ]:
cases = [
    ("version",  "version: 3.10",        'version: "3.10"'),
    ("country",  "country: no",          'country: "no"'),
    ("zip",      "zip: 01234",           'zip: "01234"'),
    ("date",     "released: 2026-10-01", 'released: "2026-10-01"'),
    ("flag",     "enabled: yes",         "enabled: true"),
]
for label, bad, good in cases:
    b = yaml.safe_load(bad)
    g = yaml.safe_load(good)
    (kb, vb), (kg, vg) = list(b.items())[0], list(g.items())[0]
    print(f"{label:<8} {bad:<24} -> {vb!r:<28} {type(vb).__name__:<8} | {good:<26} -> {vg!r} {type(vg).__name__}")

### Worked example 3.3: anchors and merge keys
`<<: *defaults` copies every key from the anchored mapping, then keys written afterwards **override** them. Predict `api` before you run the cell.

In [ ]:
doc = yaml.safe_load("""
defaults: &defaults
  timeout: 5
  retries: 3
  tls: true

api:
  <<: *defaults
  timeout: 20
  host: api.bytown.example

batch:
  <<: *defaults
  retries: 0
""")
print("api  :", doc["api"])
print("batch:", doc["batch"])

### Warm-up 3 (not graded)
Without running it first, decide what `yaml.safe_load("a: 1\nb: '1'\nc: 1.0\nd: [1, '1']\ne:\n")` returns, then check.

**Expected output**
```
{'a': 1, 'b': '1', 'c': 1.0, 'd': [1, '1'], 'e': None}
```

In [ ]:
# warm-up: your code here

### Q6 · The YAML type trap *(7 pts)*
*Focus: scalars, quoting, `safe_load`.*

A colleague wrote `risky_yaml`. Loaded as is, several values have the wrong type.

1. Load `risky_yaml` into `before` and print each key with its value and type (same style as example 3.1).
2. Write a corrected string `fixed_yaml` that loads into **exactly** these Python values and types:

| key | required value | type |
|---|---|---|
| `service` | `"billing"` | `str` |
| `version` | `"3.10"` | `str` |
| `country` | `"no"` | `str` |
| `zip` | `"01234"` | `str` |
| `deployed` | `"2026-10-01"` | `str` |
| `ssl` | `True` | `bool` (write it as `true` so every YAML parser agrees) |
| `replicas` | `3` | `int` |
| `ports` | `[80, 443]` | list of `int` |

3. Load it into `after` and print the types again.
4. In `trap_answer`, write one sentence explaining why `yaml.safe_load("zip: 01234")` does **not** give the integer `1234`.

In [ ]:
risky_yaml = """
service: billing
version: 3.10
country: no
zip: 01234
deployed: 2026-10-01
ssl: yes
replicas: 3
ports: [80, 443]
"""

before = None            # your code here
# print each key, value and type of `before`

fixed_yaml = """
"""                      # your code here: rewrite the document with the fixes

after = None             # your code here
# print each key, value and type of `after`

trap_answer = ""         # one sentence

In [ ]:
selfcheck("Q6",
    ("before is a dict", lambda: isinstance(before, dict)),
    ("after has the same keys as before", lambda: set(after) == set(before)),
    ("version, country, zip, deployed are str", lambda: all(isinstance(after[k], str) for k in ("version", "country", "zip", "deployed"))),
    ("exact string values", lambda: (after["version"], after["country"], after["zip"], after["deployed"]) == ("3.10", "no", "01234", "2026-10-01")),
    ("ssl is True (bool) and 'true' is used in the text", lambda: after["ssl"] is True and "true" in fixed_yaml),
    ("replicas int, ports list of int", lambda: after["replicas"] == 3 and after["ports"] == [80, 443]),
    ("trap_answer is a sentence", lambda: isinstance(trap_answer, str) and len(trap_answer.split()) >= 6),
)

### Worked example 3.4: moving between formats
All three libraries produce and consume the same Python structures, so converting is "load with one, dump with another". Two things to observe in the output: YAML to JSON works for any data that JSON can represent, and going to XML requires **you** to decide the shape.

In [ ]:
doc = yaml.safe_load("""
site: bytown
polling: {interval: 30, timeout: 5}
hosts: [OTT-SW-01, OTT-SW-02]
""")

as_json = json.dumps(doc, indent=2, sort_keys=True)
back = json.loads(as_json)
as_yaml = yaml.safe_dump(back, sort_keys=False)

print(as_json)
print("equal after YAML->JSON->dict?", back == doc)
print(as_yaml)

# One of many reasonable XML shapes for the same data:
root = ET.Element("config", site=doc["site"])
poll = ET.SubElement(root, "polling")
for k, v in doc["polling"].items():
    ET.SubElement(poll, k).text = str(v)
hosts = ET.SubElement(root, "hosts")
for h in doc["hosts"]:
    ET.SubElement(hosts, "host").text = h
ET.indent(root)
print(ET.tostring(root, encoding="unicode"))

### Q7 · Format conversion *(6 pts)*
*Focus: YAML, JSON and XML conversion; anchors and merge keys.*

Use `services_yaml` below.

1. Load it into `config`. Set `web_replicas`, `worker_replicas` and `worker_restart` from `config` (the `worker` has no `replicas` or `restart` of its own: they arrive through the merge key).
2. Write `yaml_to_json(text)` returning a JSON string with `indent=2` and `sort_keys=True`, and `json_to_yaml(text)` returning a YAML string with `sort_keys=False`.
3. Set `roundtrip_equal` to whether `yaml.safe_load(json_to_yaml(yaml_to_json(services_yaml)))` equals `config`.
4. Build `web_xml`: an XML **string** for the `web` mapping, with root `<web>`, one child element per scalar key (text via `str()`), and for `ports` a `<ports>` element containing one `<item>` per port. Parse it back with `ET.fromstring` into `web_root` and set `replicas_text_type` to `type(...)` of the text of its `<replicas>` element (this shows what XML does to numbers).

In [ ]:
services_yaml = """
defaults: &defaults
  image_pull: always
  restart: unless-stopped
  replicas: 1

web:
  <<: *defaults
  image: nginx:1.27
  replicas: 3
  ports: [80, 443]

worker:
  <<: *defaults
  image: python:3.12
"""

config = None
web_replicas = None
worker_replicas = None
worker_restart = None

def yaml_to_json(text):
    # your code here
    pass

def json_to_yaml(text):
    # your code here
    pass

roundtrip_equal = None
web_xml = None
web_root = None
replicas_text_type = None
# your code here

print(web_replicas, worker_replicas, worker_restart, roundtrip_equal, replicas_text_type)
print(web_xml)

In [ ]:
selfcheck("Q7",
    ("config loaded with merge keys applied", lambda: config["worker"]["image_pull"] == "always"),
    ("web_replicas is 3, worker_replicas is 1", lambda: (web_replicas, worker_replicas) == (3, 1)),
    ("worker_restart is 'unless-stopped'", lambda: worker_restart == "unless-stopped"),
    ("yaml_to_json returns sorted, indented JSON text", lambda: isinstance(yaml_to_json("b: 1\na: 2"), str) and yaml_to_json("b: 1\na: 2").index('"a"') < yaml_to_json("b: 1\na: 2").index('"b"') and "\n" in yaml_to_json("b: 1\na: 2")),
    ("json_to_yaml keeps order", lambda: json_to_yaml('{"b": 1, "a": 2}').splitlines()[0].startswith("b:")),
    ("roundtrip_equal is True", lambda: roundtrip_equal is True),
    ("web_root has <ports> with two <item> children", lambda: web_root.tag == "web" and [i.text for i in web_root.find("ports").findall("item")] == ["80", "443"]),
    ("XML numbers are text", lambda: replicas_text_type is str),
)

---
# Part 4. Calling APIs with `requests`

Quick reminders from lecture sections 19 and 20:

- `requests.get/post/put/patch/delete(url, params=, json=, headers=, timeout=)` return a `Response`.
- **Always pass `timeout=`.** The default is to wait forever.
- A 404 or 500 does **not** raise. Check `r.status_code` / `r.ok`, or call `r.raise_for_status()`.
- `json=` sends a JSON body and sets `Content-Type`. `params=` builds the query string safely.
- Status classes: 2xx success · 4xx you are wrong (fix, don't retry) · 5xx they are broken (maybe retry).

The practice API (`BASE`) has the endpoints listed in lecture 19.7. Call `reset_api()` whenever you want the original 11 devices back.

### Worked example 4.1: looking at one response closely
Run it, then change `3` to `99` and run it again to see how a 404 differs.

In [ ]:
reset_api()
r = requests.get(f"{BASE}/api/devices/3", timeout=5)

print("status_code   :", r.status_code, r.reason)
print("ok            :", r.ok)
print("content-type  :", r.headers["Content-Type"])
print("body (text)   :", r.text)
print("body (parsed) :", r.json())
print("hostname      :", r.json()["hostname"])
print("request URL   :", r.url)
print("request header:", r.request.headers["User-Agent"])

### Worked example 4.2: filters and parameters
Two filters combine with AND. Watch `r.url` to see how `params=` turns a dictionary into a query string.

In [ ]:
def show(params):
    r = requests.get(f"{BASE}/api/devices", params=params, timeout=5)
    names = [d["hostname"] for d in r.json()["results"]]
    print(f"{r.url.split('/api')[1]:<48} total={r.json()['total']:<3} {names}")

show({"per_page": 50})
show({"per_page": 50, "type": "switch"})
show({"per_page": 50, "status": "down"})
show({"per_page": 50, "type": "switch", "status": "down"})
show({"per_page": 50, "type": "toaster"})

### Worked example 4.3: one API, three representations
The same inventory is available as JSON, XML and YAML. You already know how to parse each one. This is what "the format is just the body" means.

In [ ]:
j = requests.get(f"{BASE}/api/devices", params={"per_page": 50}, timeout=5).json()["results"]
x = ET.fromstring(requests.get(f"{BASE}/api/devices.xml", timeout=5).text)
y = yaml.safe_load(requests.get(f"{BASE}/api/devices.yaml", timeout=5).text)["devices"]

print("counts   :", len(j), len(x.findall("device")), len(y))
print("hostnames equal across formats?", [d["hostname"] for d in j] == [d.findtext("hostname") for d in x.findall("device")] == [d["hostname"] for d in y])
print("ports types: JSON", type(j[0]["ports"]).__name__, "| XML", type(x.find("device").findtext("ports")).__name__, "| YAML", type(y[0]["ports"]).__name__)

### Warm-up 4 (not graded)
Using only `requests.get` and `params=`, find the hostnames of all **firewalls** and print how many there are.

**Expected output**
```
['TOR-FW-01', 'TOR-FW-02']
2
```

In [ ]:
# warm-up: your code here

### Q8 · Query the inventory *(7 pts)*
*Focus: `GET`, `params=`, status codes, `raise_for_status`.*

`target_type` is chosen from your seed. Write:

- `list_hostnames(device_type=None, status=None)`: calls `GET /api/devices` with `per_page=50` and **only the filters that are not `None`** (build the `params` dict conditionally), calls `raise_for_status()`, and returns the list of hostnames.
- `device_exists(device_id)`: returns `True` for a 200, `False` for a 404, and raises (`raise_for_status`) for anything else.

Then set:

- `matches`: hostnames of devices with `type == target_type`
- `matches_down`: the same, but only devices whose status is `"down"`
- `query_url`: the exact URL string that `requests` sent for the `matches_down` query (use `r.url` from an explicit call)
- `missing_id = seed + 100`, `missing_exists` and `present_exists` (for device `1`)

In [ ]:
reset_api()
type_choices = ["router", "switch", "firewall", "access-point"]
target_type = type_choices[seed % 4]
print("target_type:", target_type)

def list_hostnames(device_type=None, status=None):
    # your code here
    pass

def device_exists(device_id):
    # your code here
    pass

matches = None
matches_down = None
query_url = None
missing_id = seed + 100
missing_exists = None
present_exists = None
# your code here

print(matches, matches_down, query_url, missing_exists, present_exists, sep="\n")

In [ ]:
selfcheck("Q8",
    ("list_hostnames() with no filters returns all 11", lambda: len(list_hostnames()) == 11),
    ("list_hostnames only sends filters that are set", lambda: list_hostnames(status="down") == ["OTT-SW-02", "TOR-FW-02", "MTL-AP-01"]),
    ("matches is a list of str", lambda: isinstance(matches, list) and all(isinstance(h, str) for h in matches)),
    ("matches_down is a subset of matches", lambda: set(matches_down) <= set(matches)),
    ("query_url contains both filters", lambda: "type=" in query_url and "status=down" in query_url and query_url.startswith("http")),
    ("missing_exists is False, present_exists is True", lambda: missing_exists is False and present_exists is True),
)

### Worked example 4.4: the lifecycle of one record
This is the sequence Q9 asks you to repeat with your own device. Read how each call's **status code** tells you what happened, and how `Location` points to the new resource.

In [ ]:
reset_api()
demo = {"hostname": "DEMO-SW-01", "ip": "10.99.0.1", "type": "switch", "ports": 8, "tags": ["demo"]}
steps = []

r = requests.post(f"{BASE}/api/devices", json=demo, timeout=5);              steps.append(("POST", r.status_code))
demo_id = r.json()["id"]
print("created id", demo_id, "| Location:", r.headers["Location"])

r = requests.get(f"{BASE}/api/devices/{demo_id}", timeout=5);                steps.append(("GET", r.status_code))
r = requests.patch(f"{BASE}/api/devices/{demo_id}", json={"status": "down"}, timeout=5); steps.append(("PATCH", r.status_code))
print("after PATCH: status =", r.json()["status"], "| hostname untouched =", r.json()["hostname"])

r = requests.put(f"{BASE}/api/devices/{demo_id}", json={"hostname": "DEMO-RTR-01", "ip": "10.99.0.2", "type": "router"}, timeout=5); steps.append(("PUT", r.status_code))
print("after PUT  :", r.json())

r = requests.delete(f"{BASE}/api/devices/{demo_id}", timeout=5);             steps.append(("DELETE", r.status_code))
r = requests.get(f"{BASE}/api/devices/{demo_id}", timeout=5);                steps.append(("GET", r.status_code))
print(steps)

Notice the difference: `PATCH` changed only `status` and left everything else alone; `PUT` replaced the whole record, so the fields we did not send (`ports`, `tags`) reset to their defaults.

### Q9 · Device lifecycle *(9 pts)*
*Focus: `POST`, `GET`, `PATCH`, `PUT`, `DELETE`, reading status codes.*

Using `my_hostname` and `my_ip` (derived from your name and seed), do these steps **in order** against the practice API, appending each response's `status_code` to the list `codes`:

1. `POST` a new device: `hostname=my_hostname`, `ip=my_ip`, `type="switch"`, `ports=seed`, `tags=["lab"]`. Save `new_id` (from the response body) and `location` (the `Location` header).
2. `GET` it by id.
3. `PATCH` its `status` to `"down"`. Save the returned record as `patched`.
4. `PUT` a replacement: same `hostname`, `ip=f"10.99.{seed}.2"`, `type="router"`, `ports=4` (all required fields must be present). Save the returned record as `replaced`.
5. `DELETE` it.
6. `GET` it again.

Finally set `total_after` to the total number of devices the API reports (`GET /api/devices`, field `total`). `codes` must come out as `[201, 200, 200, 200, 204, 404]`.

In [ ]:
reset_api()
my_hostname = f"LAB-{your_name.split()[0].upper()}-{seed:02d}"
my_ip = f"10.99.{seed}.1"

codes = []
new_id = None
location = None
patched = None
replaced = None
total_after = None
# your code here

print(codes)
print(new_id, location)
print(patched)
print(replaced)
print(total_after)

In [ ]:
selfcheck("Q9",
    ("codes == [201, 200, 200, 200, 204, 404]", lambda: codes == [201, 200, 200, 200, 204, 404]),
    ("new_id is an int", lambda: isinstance(new_id, int)),
    ("location ends with the new id", lambda: str(location).endswith(f"/{new_id}")),
    ("patched has status 'down' and kept the hostname", lambda: patched["status"] == "down" and patched["hostname"] == my_hostname),
    ("replaced is now a router with the new ip", lambda: replaced["type"] == "router" and replaced["ip"] == f"10.99.{seed}.2" and replaced["ports"] == 4),
    ("the device is gone: total back to 11", lambda: total_after == 11),
)

### Worked example 4.5: reading error responses and authentication
A failed call is *information*. Print the status **and the body**, because the body usually says exactly what to fix. The last two lines show how the two authentication failures differ.

In [ ]:
def attempt(label, fn):
    try:
        r = fn()
        print(f"{label:<28} {r.status_code}  {r.text.strip()[:70]}")
    except requests.RequestException as e:
        print(f"{label:<28} {type(e).__name__}")

attempt("unknown id",                lambda: requests.get(f"{BASE}/api/devices/999", timeout=3))
attempt("unknown path",              lambda: requests.get(f"{BASE}/api/nothing", timeout=3))
attempt("wrong method (PUT on list)", lambda: requests.put(f"{BASE}/api/devices", json={}, timeout=3))
attempt("body not JSON",             lambda: requests.post(f"{BASE}/api/devices", data="oops", timeout=3))
attempt("JSON missing fields",       lambda: requests.post(f"{BASE}/api/devices", json={"hostname": "x"}, timeout=3))
attempt("no credentials",            lambda: requests.get(f"{BASE}/api/secure/ping", timeout=3))
attempt("wrong credentials",         lambda: requests.get(f"{BASE}/api/secure/ping", headers={"Authorization": "Bearer nope"}, timeout=3))
attempt("good credentials",          lambda: requests.get(f"{BASE}/api/secure/ping", headers={"Authorization": f"Bearer {os.environ['NETOPS_TOKEN']}"}, timeout=3))
attempt("too slow (timeout=0.5)",    lambda: requests.get(f"{BASE}/api/slow", timeout=0.5))
attempt("nobody listening",          lambda: requests.get("http://127.0.0.1:9/", timeout=1))

### Q10 · Errors and authentication *(7 pts)*
*Focus: 4xx status codes, bearer tokens, timeouts, exceptions.*

Fill the dictionary `observed` with what the practice API returns. For the first five keys store the **integer status code**; for `"timeout"` store the **name of the exception class** you caught (`type(e).__name__`).

| key | request |
|---|---|
| `no_token` | `GET /api/secure/ping` with no `Authorization` header |
| `bad_token` | the same, with `Authorization: Bearer wrong` |
| `good_token` | the same, with the real token read from `os.environ["NETOPS_TOKEN"]` |
| `bad_json` | `POST /api/devices` with `data="not json"` |
| `missing_field` | `POST /api/devices` with `json={"hostname": "x"}` |
| `timeout` | `GET /api/slow` with `timeout=0.5`, caught with `except requests.Timeout` |

Also set `error_message` to the `"error"` text from the body of the `missing_field` response, and `why_403` to **one sentence** explaining the difference between a 401 and a 403.

In [ ]:
reset_api()
observed = {}
error_message = None
why_403 = ""
# your code here

for k, v in observed.items():
    print(f"{k:<14}", v)
print(error_message)

In [ ]:
selfcheck("Q10",
    ("no_token is 401", lambda: observed["no_token"] == 401),
    ("bad_token is 403", lambda: observed["bad_token"] == 403),
    ("good_token is 200", lambda: observed["good_token"] == 200),
    ("bad_json is 400 and missing_field is 422", lambda: (observed["bad_json"], observed["missing_field"]) == (400, 422)),
    ("timeout holds an exception class name", lambda: observed["timeout"] in ("ReadTimeout", "ConnectTimeout", "Timeout")),
    ("error_message names the missing fields", lambda: "ip" in error_message and "type" in error_message),
    ("why_403 is a sentence", lambda: isinstance(why_403, str) and len(why_403.split()) >= 8),
)

---
# Part 5. Robust API clients

Real APIs are slow, flaky, rate limited and paginated. A script that works against a quiet test server and fails on the real one is not finished. Quick reminders from lecture 20.9 to 20.13:

- **Retry only transient failures**: connection errors, timeouts, 500/502/503/504, and 429 after waiting. Never retry 400/401/403/404/422: the request is wrong, and sending it again changes nothing.
- Every retry loop needs a **limit**, a **delay**, and the delay should **grow** (exponential backoff).
- Honour **`Retry-After`** on 429 instead of guessing.
- A list endpoint returns **one page at a time**. Follow `next` until it is `null`.

### Worked example 5.1: a retry loop that logs what it did
Study how this function decides *whether* to retry (the `if` on the status code), *how long* to wait (doubling), and *when to give up* (the last attempt re-raises). Q11 asks you to write something very close to it.

In [ ]:
def fetch_with_log(url, attempts=4, base_delay=0.1):
    """GET url, retrying 5xx and network errors. Returns (response, log)."""
    log = []
    for attempt in range(1, attempts + 1):
        try:
            r = requests.get(url, timeout=2)
        except (requests.ConnectionError, requests.Timeout) as e:
            log.append((attempt, type(e).__name__))
            outcome_retryable = True
        else:
            log.append((attempt, r.status_code))
            if r.status_code < 500:
                return r, log                       # success, or a 4xx we must not retry
            outcome_retryable = True

        if attempt == attempts:
            raise RuntimeError(f"gave up after {attempts} attempts: {log}")
        time.sleep(base_delay * 2 ** (attempt - 1))

reset_api()
r, log = fetch_with_log(f"{BASE}/api/flaky")
print("log   :", log)
print("final :", r.status_code, r.json())

r, log = fetch_with_log(f"{BASE}/api/devices/999")
print("log   :", log, "<- a 404 is returned immediately, never retried")

### Worked example 5.2: waiting for `Retry-After`
`/api/limited` answers 429 on every third call, and says how long to wait. The loop makes 6 requests and waits only when it is told to.

In [ ]:
reset_api()
for n in range(1, 7):
    r = requests.get(f"{BASE}/api/limited", timeout=3)
    if r.status_code == 429:
        wait = int(r.headers["Retry-After"])
        print(f"request {n}: 429, server asks us to wait {wait}s ... waiting")
        time.sleep(wait)
        r = requests.get(f"{BASE}/api/limited", timeout=3)
    print(f"request {n}: {r.status_code}")

### Warm-up 5 (not graded)
Predict the list of status codes you would see if you made 5 plain `requests.get` calls to `/api/flaky` right after `reset_api()` with **no retrying at all**, then confirm.

**Expected output**
```
[503, 503, 200, 200, 200]
```

In [ ]:
# warm-up: your code here

### Q11 · Retry and rate limits *(9 pts)*
*Focus: backoff, transient versus permanent errors, `Retry-After`.*

**(a)** Write `get_json_with_retry(url, attempts=4, base_delay=0.1)`:

- send `GET` with `timeout=2`;
- after **every** attempt append `(attempt_number, outcome)` to the global list `call_log`, where `outcome` is the integer status code, or the exception class name if no response arrived;
- retry (after `time.sleep(base_delay * 2 ** (attempt - 1))`) on `requests.ConnectionError`, `requests.Timeout`, and status 500, 502, 503 or 504;
- for any other 4xx, call `raise_for_status()` immediately (**no retry**);
- on success return the parsed JSON; if the last attempt still fails, re-raise (for a 5xx use `raise_for_status()`).

Run it on `/api/flaky` (after `reset_api()`) and on `/api/devices/999`, clearing `call_log` before each, and keep copies as `flaky_log` and `missing_log`. Catch the `requests.HTTPError` from the second call into `missing_error`.

**(b)** `calls_n` is chosen from your seed. After `reset_api()`, make `calls_n` calls to `/api/limited`. Whenever a call returns 429, sleep for the `Retry-After` value and call once more. Record each call's **final** status in `statuses`, and the number of times you had to wait in `waits`.

In [ ]:
call_log = []

def get_json_with_retry(url, attempts=4, base_delay=0.1):
    # your code here
    pass

flaky_data = None
flaky_log = None
missing_log = None
missing_error = None
# your code here (a)

calls_n = seed % 4 + 4
statuses = []
waits = 0
# your code here (b)

print(flaky_data, flaky_log)
print(missing_log, type(missing_error).__name__)
print(calls_n, statuses, waits)

In [ ]:
selfcheck("Q11",
    ("flaky endpoint needed exactly 3 attempts", lambda: flaky_log == [(1, 503), (2, 503), (3, 200)]),
    ("flaky_data is the parsed JSON", lambda: isinstance(flaky_data, dict) and flaky_data.get("ok") is True),
    ("a 404 is attempted once and raises HTTPError", lambda: missing_log == [(1, 404)] and isinstance(missing_error, requests.HTTPError)),
    ("statuses has one final status per call, all 200", lambda: len(statuses) == calls_n and set(statuses) == {200}),
    ("waits matches the number of 429 responses the server sent", lambda: waits == mockapi._state["limited_calls"] - calls_n and waits >= 1),
)

### Worked example 5.3: pagination as a generator
The generator hides the page loop from the caller: the caller just writes `for device in iter_pages(...)`. Two details to notice: `params` is cleared after the first request because the `next` URL already contains the query string, and `pages_seen` records how many requests were needed.

In [ ]:
reset_api()
demo_pages = []

def iter_demo(per_page):
    url = f"{BASE}/api/devices"
    params = {"per_page": per_page}
    while url:
        r = requests.get(url, params=params, timeout=5)
        r.raise_for_status()
        page = r.json()
        demo_pages.append(page["page"])
        yield from page["results"]
        url = page["next"]            # None after the last page
        params = None

names = [d["hostname"] for d in iter_demo(per_page=4)]
print("pages fetched:", demo_pages)
print("devices      :", len(names))
print(names)

# A generator is lazy: take only the first 2 and the loop stops early.
demo_pages.clear()
first_two = []
for d in iter_demo(per_page=2):
    first_two.append(d["hostname"])
    if len(first_two) == 2:
        break
print(first_two, "-> pages fetched:", demo_pages)

### Q12 · Pagination *(8 pts)*
*Focus: following `next` links, generators, summarizing results.*

`page_size` is chosen from your seed. After `reset_api()`:

1. Write the generator `iter_devices(page_size)` that yields every device across all pages by following the `next` link, and appends each page number it fetches to the global list `pages_seen`.
2. Set `all_devices = list(iter_devices(page_size))`.
3. Compute:
   - `pages_fetched`: the number of pages requested (`len(pages_seen)`)
   - `by_type`: a plain `dict` of `{type: count}` (use `Counter`)
   - `total_ports`: the sum of `ports` over all devices
   - `down_hosts`: sorted list of hostnames with status `"down"`
   - `most_ports_hosts`: sorted list of **all** hostnames that share the maximum `ports` value

In [ ]:
reset_api()
page_size = seed % 3 + 2
pages_seen = []

def iter_devices(page_size):
    # your code here
    pass

all_devices = None
pages_fetched = None
by_type = None
total_ports = None
down_hosts = None
most_ports_hosts = None
# your code here

print("page_size:", page_size)
print(pages_fetched, len(all_devices or []))
print(by_type)
print(total_ports)
print(down_hosts)
print(most_ports_hosts)

In [ ]:
import math
selfcheck("Q12",
    ("iter_devices is a generator", lambda: hasattr(iter_devices(2), "__next__")),
    ("all 11 devices, no duplicates", lambda: len(all_devices) == 11 and len({d["id"] for d in all_devices}) == 11),
    ("pages_fetched == ceil(11 / page_size)", lambda: pages_fetched == math.ceil(11 / page_size)),
    ("pages_seen are 1..n in order", lambda: pages_seen[:pages_fetched] == list(range(1, pages_fetched + 1))),
    ("by_type is a plain dict adding up to 11", lambda: type(by_type) is dict and sum(by_type.values()) == 11),
    ("total_ports is an int", lambda: isinstance(total_ports, int)),
    ("down_hosts is sorted", lambda: down_hosts == sorted(down_hosts) and len(down_hosts) == 3),
    ("most_ports_hosts is a sorted list", lambda: most_ports_hosts == sorted(most_ports_hosts) and len(most_ports_hosts) >= 1),
)

---
# Part 6. Capstone: audit the fleet

You now have every piece. This task combines them into one small, reusable program in the style of the lecture's `InventoryClient`: a **class** (Week 4), a **custom exception** (Week 4), a **session** with defaults, a **generator** for pagination, and **three output formats** (Week 5).

### Worked example 6.1: the shape of the client
Read this skeleton. `InventoryAPIError` carries the status code. The one private method `_request` is the only place that touches `requests`, so error handling is written once. Q13 asks you to finish three methods.

In [ ]:
class InventoryAPIError(Exception):
    """Any failure talking to the inventory API. `status_code` is None for network problems."""
    def __init__(self, message, status_code=None):
        super().__init__(message)
        self.status_code = status_code


def describe_failure(exc):
    """Example of using the custom exception from the caller's side."""
    if exc.status_code is None:
        return f"network problem: {exc}"
    if 400 <= exc.status_code < 500:
        return f"our request was refused ({exc.status_code}): {exc}"
    return f"server trouble ({exc.status_code}): {exc}"

e = InventoryAPIError("device 9 not found", 404)
print(describe_failure(e))
print(describe_failure(InventoryAPIError("cannot reach host")))
print(isinstance(e, Exception), e.status_code)

### Q13 · Audit capstone *(12 pts)*
*Focus: a client class, error translation, pagination, JSON/YAML/XML output.*

**(a) Finish the `InventoryClient` class** (methods marked `# your code here`):

- `_request(method, path, **kwargs)`: send the request through `self.session` with `timeout=self.timeout`. Convert `requests.Timeout` and `requests.ConnectionError` into `InventoryAPIError(..., None)`. If the response is not `ok`, raise `InventoryAPIError(f"{method} {path} -> {status}: {error text}", status)`, using the `"error"` field of the JSON body when there is one. Otherwise return the parsed JSON, or `None` when the body is empty.
- `devices(**filters)`: a generator over all pages (use `per_page=5`, follow `next`).
- `add_tag(device_id, tag)`: `GET` the device, build a **sorted list without duplicates** of its tags plus `tag`, `PATCH` them, and return the updated record.

**(b) Run the audit.** After `reset_api()`, create `client = InventoryClient(BASE, os.environ["NETOPS_TOKEN"])`, define `audit_tag = f"audited-{your_name.split()[0].lower()}"` and, for every device whose `type == target_type` (from Q8), call `add_tag`. Build this dictionary:

```python
audit = {"audited_by": your_name, "tag": audit_tag, "type": target_type, "count": <n>,
         "devices": [ {"id": .., "hostname": .., "status": .., "tags": [...]}, ... ]}   # sorted by id, tags AFTER tagging
```

**(c) Write the audit in all three formats** into `w5lab/`:

- `audit.json` (`indent=2`, `sort_keys=True`)
- `audit.yaml` (`sort_keys=False`)
- `audit.xml`: root `<audit by=".." type=".." count="..">` with one `<device id=".." status="..">` per device holding a `<hostname>` and one `<tag>` per tag

**(d) Verify.** Set `json_equals_yaml` (reload both files; are the two Python structures equal?), `xml_count` (number of `<device>` elements read back from `audit.xml`), `api_confirms` (`True` if asking the API again shows `audit_tag` on every audited device), and `error_status`: call `client.add_tag(9999, "x")` inside `try`/`except InventoryAPIError` and store `e.status_code`.

In [ ]:
class InventoryClient:
    def __init__(self, base_url, token, timeout=5):
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self.session = requests.Session()
        self.session.headers.update({"Accept": "application/json", "Authorization": f"Bearer {token}"})

    def _request(self, method, path, **kwargs):
        # your code here
        pass

    def devices(self, **filters):
        # your code here (generator)
        pass

    def add_tag(self, device_id, tag):
        # your code here
        pass


reset_api()
client = InventoryClient(BASE, os.environ["NETOPS_TOKEN"])
audit_tag = f"audited-{your_name.split()[0].lower()}"

audit = None
# your code here (b): tag every device of target_type and build `audit`

audit_json = work / "audit.json"
audit_yaml = work / "audit.yaml"
audit_xml = work / "audit.xml"
# your code here (c): write the three files

json_equals_yaml = None
xml_count = None
api_confirms = None
error_status = None
# your code here (d)

print(audit)
print(json_equals_yaml, xml_count, api_confirms, error_status)

In [ ]:
selfcheck("Q13",
    ("InventoryClient.devices is a generator", lambda: hasattr(client.devices(), "__next__")),
    ("audit has the required keys", lambda: set(audit) == {"audited_by", "tag", "type", "count", "devices"}),
    ("count matches the number of devices listed", lambda: audit["count"] == len(audit["devices"]) >= 1),
    ("every audited device has the tag, tags are sorted without duplicates", lambda: all(audit_tag in d["tags"] and d["tags"] == sorted(set(d["tags"])) for d in audit["devices"])),
    ("devices sorted by id", lambda: [d["id"] for d in audit["devices"]] == sorted(d["id"] for d in audit["devices"])),
    ("three files written", lambda: audit_json.exists() and audit_yaml.exists() and audit_xml.read_text(encoding="utf-8").startswith("<?xml")),
    ("JSON file is sorted, YAML keeps order", lambda: audit_json.read_text(encoding="utf-8").splitlines()[1].strip().startswith('"audited_by"') and audit_yaml.read_text(encoding="utf-8").startswith("audited_by:")),
    ("json_equals_yaml is True", lambda: json_equals_yaml is True),
    ("xml_count equals count", lambda: xml_count == audit["count"]),
    ("api_confirms is True", lambda: api_confirms is True),
    ("error_status is 404", lambda: error_status == 404),
)

---
## Submit

1. Make sure `your_name` and `seed` at the top are filled in with **your own** values.
2. Choose **Restart & Run All** (Colab: *Runtime > Restart session and run all*). Confirm there are **no red tracebacks** from top to bottom. A self-check line that reports a failed check is not a traceback, but read it: it is telling you something is wrong.
3. Check that every `selfcheck` cell prints "all ... checks passed" (or that you understand why one does not).
4. Save, then rename the file to `W05_Lab_<your-last-name>.ipynb` (for example `W05_Lab_Moeini.ipynb`).
5. Upload **only** that `.ipynb` file to **Week 5 Lab Submission**. Do not upload `mockapi.py` or the `w5lab/` folder: the notebook recreates both when it runs.

**Late policy:** submissions received after the deadline receive a grade of 0.

Run the last cell to stop the practice API before you finish.

In [ ]:
mockapi.stop()
print("practice API stopped")